### Import

In [1]:
import pandas as pd
import datetime as dt

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
mimiciv = "PATH TO DATA/mimiciv(3.1)/"

# Hosp

### hosp - patients

dod: out of hospital mortality for MIMIC-IV patients up to one year post-hospital discharge. 

In [3]:
patients = pd.read_csv(mimiciv + "hosp/patients.csv")

In [68]:
patients.head(3)

In [5]:
print("# all patients = ", patients.subject_id.nunique())
print("# patients alive after one year = ", patients[patients.dod.isnull()].shape[0])
print("# patients dead after one year = ", patients[patients.dod.notnull()].shape[0])
print("# patients over 89 = ", patients[patients.anchor_age > 89].shape[0])
print(patients.gender.unique())

# all patients =  364627
# patients alive after one year =  326326
# patients dead after one year =  38301
# patients over 89 =  8445
['F' 'M']


### hosp - transfer

In [6]:
transfers = pd.read_csv(mimiciv + "hosp/transfers.csv")

In [7]:
transfers.head(3)

In [8]:
print("# all patients = ", transfers.subject_id.nunique())
print("# all admissions to hospital = ", transfers.hadm_id.nunique())
print("# all transfers = ", transfers.transfer_id.nunique())

# all patients =  364627
# all admissions to hospital =  546024
# all transfers =  2413581


### hosp - admissions

In [9]:
admissions = pd.read_csv(mimiciv + "hosp/admissions.csv")

In [10]:
admissions.head(3)

In [11]:
print("# hospital admissions = ", admissions.hadm_id.nunique())
print("# unique patients = ", admissions.subject_id.nunique())
print("# dead at hospital discharge= ", admissions[admissions.hospital_expire_flag == 1].shape[0])
print("# alive at hospital discharge = ", admissions[admissions.hospital_expire_flag == 0].shape[0])

# hospital admissions =  546028
# unique patients =  223452
# dead at hospital discharge=  11801
# alive at hospital discharge =  534227


### hosp - diagnoses_icd

In [12]:
diagnose_icd = pd.read_csv(mimiciv + "hosp/diagnoses_icd.csv")

In [13]:
diagnose_icd.head(3)

In [14]:
print("# all patients = ", diagnose_icd.subject_id.nunique())
print("# all hospital admissions = ", diagnose_icd.hadm_id.nunique())
print("# unique ICD-9  = ", diagnose_icd[diagnose_icd.icd_version == 9].icd_code.nunique())
print("# unique ICD-10 = ", diagnose_icd[diagnose_icd.icd_version == 10].icd_code.nunique())

# all patients =  223291
# all hospital admissions =  545497
# unique ICD-9  =  9143
# unique ICD-10 =  19440


In [15]:
d_diagnose_icd = pd.read_csv(mimiciv + "hosp/d_icd_diagnoses.csv")

In [16]:
d_diagnose_icd.head(3)

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"


In [17]:
print("# unique ICD-9  = ", d_diagnose_icd[d_diagnose_icd.icd_version == 9].icd_code.nunique())
print("# unique ICD-10 = ", d_diagnose_icd[d_diagnose_icd.icd_version == 10].icd_code.nunique())

# unique ICD-9  =  14666
# unique ICD-10 =  97441


### hosp - procedures_icd

Procedures during the hospital stay can be billed (1) by the hospital or (2) by the provider. 
This table contains only procedures billed by the hospital.

The date of the associated procedures. Date does not strictly correlate with seq_num.

In [18]:
procedure_icd = pd.read_csv(mimiciv + "hosp/procedures_icd.csv")

In [19]:
procedure_icd.head(3)

In [20]:
print("# all patients = ", procedure_icd.subject_id.nunique())
print("# all hospital admissions = ", procedure_icd.hadm_id.nunique())
print("# unique ICD-9  = ", procedure_icd[procedure_icd.icd_version == 9].icd_code.nunique())
print("# unique ICD-10 = ", procedure_icd[procedure_icd.icd_version == 10].icd_code.nunique())

# all patients =  150711
# all hospital admissions =  287504
# unique ICD-9  =  2557
# unique ICD-10 =  12354


In [21]:
d_procedure_icd = pd.read_csv(mimiciv + "hosp/d_icd_procedures.csv")

In [22]:
d_procedure_icd.head(3)

,icd_code,icd_version,long_title
0,0001,9,Therapeutic ultrasound of vessels of head and neck
1,0002,9,Therapeutic ultrasound of heart
2,0003,9,Therapeutic ultrasound of peripheral vascular vessels


In [23]:
print("# unique ICD-9  = ", d_procedure_icd[d_procedure_icd.icd_version == 9].icd_code.nunique())
print("# unique ICD-10 = ", d_procedure_icd[d_procedure_icd.icd_version == 10].icd_code.nunique())

# unique ICD-9  =  3888
# unique ICD-10 =  82535


### Surgery using Procedure

In [24]:
icustays = pd.read_csv('PATH TO DATA/mimiciv_v1/Data/EHR/demographic_all.csv')
icustays = icustays[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']]

In [25]:
procedure = procedure_icd.merge(d_procedure_icd, on=['icd_code', 'icd_version'], how='left')
procedure = procedure[['subject_id', 'hadm_id', 'chartdate', 'icd_code', 'icd_version', 'long_title']]

In [26]:
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
procedure['chartdate'] = pd.to_datetime(procedure['chartdate'])

In [27]:
procedure = procedure.merge(icustays, on=['subject_id', 'hadm_id'], how='left')
procedure = procedure[procedure.stay_id.notnull()]

In [28]:
procedure = procedure[((procedure['chartdate'] >= (procedure['intime'] - pd.Timedelta(hours=12))) & (procedure['chartdate'] < procedure['outtime']))]
procedure = procedure[['subject_id', 'hadm_id', 'stay_id', 'icd_code', 'icd_version', 'long_title']]

In [29]:
icd_data_json = list(procedure.long_title.unique())

In [30]:
surgery_keywords = ["surgery", "resection", "excision", "repair", "incision", "bypass", 
                    "fusion", "removal", "replacement", "transplant", "anastomosis", 
                    "amputation", "graft", "insertion", "extraction", "ligation",
                    "detachment", "arthroscopy", "laparoscopy", "debridement", "ostomy",
                    "arthrodesis", "arthroplasty", "implantation", "transplantation",
                    "reconstruction", "endarterectomy"]

In [31]:
surgery_procedures = [procedure for procedure in icd_data_json if any(keyword.lower() in procedure.lower() for keyword in surgery_keywords)]
procedure = procedure[procedure.long_title.isin(surgery_procedures)]

In [32]:
procedure.head()

### hosp - microbiologyevents

In [50]:
microevents = pd.read_csv(mimiciv + "hosp/microbiologyevents.csv")

<ipython-input>:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  microevents = pd.read_csv(mimiciv + "hosp/microbiologyevents.csv")


In [51]:
microevents.head(1)

In [52]:
print("# all micro test orders = ", microevents.microevent_id.nunique())
print("# unique patients = ", microevents.subject_id.nunique())
print("# unique hospital admissions = ", microevents.hadm_id.nunique())
print("# unique samples (specimens) = ", microevents.micro_specimen_id.nunique())
print("# rows with missing hospital admissions = ", microevents[microevents.hadm_id.isnull()].shape[0])

# all micro test orders =  3988224
# unique patients =  222313
# unique hospital admissions =  201096
# unique samples (specimens) =  1924289
# rows with missing hospital admissions =  2228343


### hosp - prescriptions

The prescriptions and pharmacy tables are intended to be used together: prescriptions contains the order made by a provider and pharmacy stores detailed information regarding the compound prescribed.

The eMAR system was deployed between 2014–2016, and thus all hospitalizations from 2016 onward would be anticipated to have records within eMAR.

In [53]:
prescriptions = pd.read_csv(mimiciv + "hosp/prescriptions.csv")

<ipython-input>:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv(mimiciv + "hosp/prescriptions.csv")


In [54]:
prescriptions.head(3)

In [55]:
print("# table size = ", prescriptions.shape)
print("# unique patients = ", prescriptions.subject_id.nunique())
print("# unique hospital admissions = ", prescriptions.hadm_id.nunique())
print("# unique id to link administrations in emar to pharmacy table = ", prescriptions.pharmacy_id.nunique())
print("# unique medication name = ", prescriptions.drug.nunique())
print("# unique Generic Sequence Number (GSN) = ", prescriptions.gsn.nunique())
print("# unique National Drug Code (NDC) = ", prescriptions.ndc.nunique())

# table size =  (20292611, 21)
# unique patients =  196738
# unique hospital admissions =  463328
# unique id to link administrations in emar to pharmacy table =  17779025
# unique medication name =  10926
# unique Generic Sequence Number (GSN) =  7928
# unique National Drug Code (NDC) =  6588


### hosp - pharmacy

In [56]:
pharmacy = pd.read_csv(mimiciv + "hosp/pharmacy.csv")

<ipython-input>:1: DtypeWarning: Columns (16,18,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  pharmacy = pd.read_csv(mimiciv + "hosp/pharmacy.csv")


In [57]:
pharmacy.head(3)

In [58]:
print("# table size = ", pharmacy.shape)
print("# unique patients = ", pharmacy.subject_id.nunique())
print("# unique hospital admissions = ", pharmacy.hadm_id.nunique())
print("# unique medication name = ", pharmacy.medication.nunique())

# table size =  (17847567, 27)
# unique patients =  196738
# unique hospital admissions =  463328
# unique medication name =  11058


### hosp - labevent

In [59]:
labevents = pd.read_csv(mimiciv + "hosp/labevents.csv")

In [60]:
labevents.head(3)

In [61]:
print("# table size = ", labevents.shape)
print("# unique patients = ", labevents.subject_id.nunique())
print("# unique hospital admissions = ", labevents.hadm_id.nunique())
print("# unique medication name = ", labevents.itemid.nunique())

# table size =  (158374764, 16)
# unique patients =  313442
# unique hospital admissions =  447689
# unique medication name =  976


### hosp - d_labitems

In [62]:
d_labitems = pd.read_csv(mimiciv + "hosp/d_labitems.csv")

In [63]:
d_labitems.head(3)

,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas


In [64]:
print("# unique lab item = ", d_labitems.itemid.nunique())
print("# unique lab name = ", d_labitems.label.nunique())
print("# unique lab substance = ", d_labitems.fluid.nunique())
print("# unique lab category = ", d_labitems.category.nunique())

# unique lab item =  1650
# unique lab name =  1191
# unique lab substance =  12
# unique lab category =  3


# ICU

### icu - icustays

In [65]:
icustays = pd.read_csv(mimiciv + "icu/icustays.csv")

In [66]:
icustays.head(3)

In [72]:
print("# unique patients = ", icustays.subject_id.nunique())
print("# unique hospital admissions = ", icustays.hadm_id.nunique())
print("# unique icu admissions = ", icustays.stay_id.nunique())
print("# unique icu transfered stays = ", icustays[icustays.first_careunit != icustays.last_careunit].stay_id.nunique())

# unique patients =  65366
# unique hospital admissions =  85242
# unique icu admissions =  94458
# unique icu transfered stays =  0


### icu - d_items

In [73]:
d_items = pd.read_csv(mimiciv + "icu/d_items.csv")

In [74]:
d_items.head(3)

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN


In [75]:
print("# unique items = ", d_items.itemid.nunique())
print("# unique label name = ", d_items.label.nunique())
print("# unique categories = ", d_items.category.nunique())
print("links to other tables = ", d_items.linksto.unique())
print("types of itemes = ", d_items.param_type.unique())

# unique items =  4095
# unique label name =  3965
# unique categories =  83
links to other tables =  ['chartevents' 'datetimeevents' 'ingredientevents' 'inputevents'
 'procedureevents' 'outputevents']
types of itemes =  ['Text' 'Date and time' 'Numeric' 'Ingredient' 'Solution' 'Processes'
 'Checkbox' 'Numeric with tag']


### icu - chartevent

In [76]:
chartevents = pd.read_csv(mimiciv + "icu/chartevents.csv", chunksize=100)

In [77]:
next(chartevents).head(3)

### icu - datetimeevents

In [78]:
datetimeevents = pd.read_csv(mimiciv + "icu/datetimeevents.csv")

In [79]:
datetimeevents.head(3)

In [80]:
print("# unique patients = ", datetimeevents.subject_id.nunique())
print("# unique hospital admissions = ", datetimeevents.hadm_id.nunique())
print("# unique icu admissions = ", datetimeevents.stay_id.nunique())
print("# unique items = ", datetimeevents.itemid.nunique())
print("# unique types = ", datetimeevents.valueuom.unique())

# unique patients =  65097
# unique hospital admissions =  84804
# unique icu admissions =  93601
# unique items =  174
# unique types =  ['Date' 'Date and Time']


### icu - procedureevents

This table is not a required documentation field during routine care. As a result, existence of procedures here indicates their presence, but absence does not indicate the procedure was not conducted. 

In [3]:
procedureevents = pd.read_csv(mimiciv + "icu/procedureevents.csv")

In [4]:
procedureevents.head(3)

In [5]:
print("# unique patients = ", procedureevents.subject_id.nunique())
print("# unique hospital admissions = ", procedureevents.hadm_id.nunique())
print("# unique icu admissions = ", procedureevents.stay_id.nunique())
print("# unique items = ", procedureevents.itemid.nunique())
print("# unique procedure events = ", procedureevents.ordercategoryname.nunique())
print("# procedure events = ", procedureevents.ordercategoryname.unique())

# unique patients =  50843
# unique hospital admissions =  66127
# unique icu admissions =  72711
# unique items =  157
# unique procedure events =  14
# procedure events =  ['Procedures' 'Peripheral Lines' 'Ventilation' 'Communication' 'Imaging'
 'Invasive Lines' 'Intubation/Extubation' 'Tubes' 'Significant Events'
 'Dialysis' 'Continuous Procedures' 'CRRT Filter Change'
 '17 - Inhaled Meds' 'Peritoneal Dialysis']


### icu - inputevents

In [26]:
inputevents = pd.read_csv(mimiciv + "icu/inputevents.csv")

In [27]:
inputevents.head(3)

In [28]:
print("# unique patients = ", inputevents.subject_id.nunique())
print("# unique hospital admissions = ", inputevents.hadm_id.nunique())
print("# unique icu admissions = ", inputevents.stay_id.nunique())
print("# unique items = ", inputevents.itemid.nunique())

# unique patients =  50755
# unique hospital admissions =  65986
# unique icu admissions =  72690
# unique items =  324


### icu - outputs

In [29]:
outputevents = pd.read_csv(mimiciv + "icu/outputevents.csv")

In [30]:
outputevents.head(3)

In [31]:
print("# unique patients = ", outputevents.subject_id.nunique())
print("# unique hospital admissions = ", outputevents.hadm_id.nunique())
print("# unique icu admissions = ", outputevents.stay_id.nunique())
print("# unique items = ", outputevents.itemid.nunique())

# unique patients =  50198
# unique hospital admissions =  64692
# unique icu admissions =  71111
# unique items =  71
